# Challenge 04 - Speaker Recognition (GMM)

Recognition of the speaker (31 people) of each audio track using only the MFCCs. One GMM is trained per speaker and each test track is assigned to the speaker whose model fits it best. The predictions are saved in `Es6Predictions.txt`.

## Setup

In [1]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

import numpy as np
from joblib import Parallel, delayed
from scipy.signal import savgol_filter
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

# Configuration
db_path = '../../DBs/AudioDigits_ML26_27'
train_file = 'audio_ml2627_train.pkl'
test_file = 'audio_ml2627_test_no_labels.pkl'
result_path = '../Es6Predictions.txt'

# GMM hyperparameters
n_components = 32
covariance_type = 'diag'
reg_covar = 1e-2
random_state = 1234

## Helper Functions

Dataset loading, preprocessing of the MFCCs (c0 relative to the track maximum, deltas and delta-deltas, standardization), training of one GMM per speaker, scoring of the tracks and export of the predictions.

In [2]:
def load_dataset(db_path, filename):
    return np.load(os.path.join(db_path, filename), allow_pickle=True)


def delta(m, order):
    # same as librosa.feature.delta(m.T, width=9, order=order).T
    return savgol_filter(m, 9, polyorder=order, deriv=order, axis=0, mode='interp')


def preprocess(mfccs):
    tracks = []
    for m in mfccs:
        m = np.asarray(m, dtype=float).copy()
        m[:, 0] -= m[:, 0].max()
        tracks.append(np.hstack([m, delta(m, 1), delta(m, 2)]))
    return tracks


def standardize(train_tracks, test_tracks):
    scaler = StandardScaler().fit(np.vstack(train_tracks))
    return [scaler.transform(m) for m in train_tracks], [scaler.transform(m) for m in test_tracks]


def fit_gmm(frames):
    return GaussianMixture(n_components=n_components, covariance_type=covariance_type, reg_covar=reg_covar,
                           max_iter=200, random_state=random_state).fit(frames)


def train_speaker_models(tracks, speakers):
    ids = np.unique(speakers)
    gmms = Parallel(n_jobs=-1)(delayed(fit_gmm)(np.vstack([m for m, s in zip(tracks, speakers) if s == i])) for i in ids)
    return dict(zip(ids, gmms))


def predict_speaker(models, tracks):
    scores = []
    for m in tracks:
        ll = np.stack([g.score_samples(m) for g in models.values()], axis=1)
        scores.append(np.exp(ll - logsumexp(ll, axis=1, keepdims=True)).mean(axis=0))
    scores = np.array(scores)
    scores = (scores - scores.mean(axis=0)) / (scores.std(axis=0) + 1e-9)
    return np.array(list(models))[scores.argmax(axis=1)]


def save_predictions(predictions, path):
    np.savetxt(path, np.asarray(predictions, dtype=int), fmt='%i')
    print(f'Predictions successfully saved to {path}')

## Data Loading

The training set contains sessions 1 and 4 (620 tracks, 20 per speaker), the test set sessions 2 and 3 (620 tracks, without labels). Only the MFCCs are used.

In [3]:
train_data = load_dataset(db_path, train_file)
test_data = load_dataset(db_path, test_file)

train_x = preprocess(train_data['mfccs'])
train_y = np.array(train_data['person_ids'])
train_sessions = np.array(train_data['session_ids'])
test_x = preprocess(test_data['mfccs'])

print('Training tracks: %d, test tracks: %d, speakers: %d' % (len(train_x), len(test_x), len(np.unique(train_y))))
print('Frames per track: %d-%d, features per frame: %d' % (min(len(m) for m in train_x), max(len(m) for m in train_x), train_x[0].shape[1]))

Training tracks: 620, test tracks: 620, speakers: 31
Frames per track: 79-745, features per frame: 60


## Model: One GMM per Speaker

One `GaussianMixture` with 32 diagonal components per speaker. Each track is scored with the mean posterior of each speaker over its frames, and the scores of each model are standardized across the tracks before taking the best speaker.

## Model Evaluation

The split is done by session to simulate new recording conditions: train on one session and validate on the other, in both directions.

In [4]:
for train_session, val_session in [(1, 4), (4, 1)]:
    tr = train_sessions == train_session
    val = train_sessions == val_session
    x_tr, x_val = standardize([m for m, t in zip(train_x, tr) if t], [m for m, v in zip(train_x, val) if v])
    models = train_speaker_models(x_tr, train_y[tr])
    val_pred = predict_speaker(models, x_val)
    print('Train session %d -> validation session %d: accuracy %.4f' % (train_session, val_session, np.mean(val_pred == train_y[val])))

Train session 1 -> validation session 4: accuracy 0.6806
Train session 4 -> validation session 1: accuracy 0.7161


## Prediction and Export

The GMMs are retrained on all training tracks (sessions 1 and 4) and each test track is assigned to the most likely speaker. The predictions are saved in `Es6Predictions.txt`, one speaker id per line.

In [5]:
x_train, x_test = standardize(train_x, test_x)
models = train_speaker_models(x_train, train_y)
predictions = predict_speaker(models, x_test)
assert len(predictions) == 620
save_predictions(predictions, result_path)

Predictions successfully saved to ../Es6Predictions.txt
